# Phase B Session 3: Full SU-HBO (v2)
## Using real J_topo computation from trained weights

**Session 2 Result**: Spearman r = 0.81 > 0.70 ✓
**Key fix**: Compute J_topo from actual network weights, not heuristics

In [1]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Done")

/kaggle/working
Cloning into 'ThermoRG-NN'...
remote: Enumerating objects: 897, done.
remote: Counting objects: 100% (291/291), done.
remote: Compressing objects: 100% (223/223), done.
remote: Total 897 (delta 123), reused 158 (delta 65), pack-reused 606 (from 1)
Receiving objects: 100% (897/897), 1.57 MiB | 18.50 MiB/s, done.
Resolving deltas: 100% (445/445), done.
/kaggle/working/ThermoRG-NN
Done


In [2]:
import sys
sys.path.insert(0, '/kaggle/working/ThermoRG-NN/src')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import json, os

from thermorg_suhbo import (
    compute_utility,
    DEFAULT_K, DEFAULT_B, DEFAULT_GAMMA_C, DEFAULT_LAMBDA,
)

print(f"Imports OK. Defaults: lambda={DEFAULT_LAMBDA}, k={DEFAULT_K}, B={DEFAULT_B}, gamma_c={DEFAULT_GAMMA_C}")

Imports OK. Defaults: lambda=10.0, k=0.06, B=0.15, gamma_c=2.0


In [3]:
# Load Session 2 calibration indices
s2_path = '/kaggle/working/ThermoRG-NN/experiments/phase_b/phase_b_session2_results.json'
if os.path.exists(s2_path):
    with open(s2_path) as f:
        s2 = json.load(f)
    calib_idx = set(s2['calib_indices'])
    print(f"Loaded calibration indices: {len(calib_idx)}")
else:
    calib_idx = set(range(5000))
    print("Using fallback calibration indices: 0-4999")

Loaded calibration indices: 2500


In [4]:
# Load CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

all_idx = set(range(len(full_train)))
main_idx = list(all_idx - calib_idx)
main_set = Subset(full_train, main_idx)

print(f"Full: {len(full_train)}, Excluded: {len(calib_idx)}, Main: {len(main_set)}")

batch_size = 128
main_loader = DataLoader(main_set, batch_size=batch_size, shuffle=True, num_workers=2)

100%|██████████| 170M/170M [00:02<00:00, 78.4MB/s]


Full: 50000, Excluded: 2500, Main: 47500


In [5]:
def compute_j_topo_from_model(model):
    """
    Compute J_topo from trained model weights using participation ratio.
    J_topo = exp(-mean|log eta_l|) where eta_l are eigenvalues of W_eff^T W_eff
    """
    eigenvalues = []
    
    for name, param in model.named_parameters():
        if 'conv' in name and 'weight' in name:
            # Get weight matrix
            w = param.data.detach()
            
            if w.dim() == 4:
                # Conv weight: (out_channels, in_channels, kH, kW)
                # Reshape to (out_channels, in_channels * kH * kW)
                w_mat = w.reshape(w.size(0), -1)
            else:
                w_mat = w
            
            # Compute W^T W
            wt_w = torch.matmul(w_mat, w_mat.T)
            
            # Compute eigenvalues
            try:
                eigvals = torch.linalg.eigvalsh(wt_w)
                eigvals = eigvals.cpu().numpy()
                eigvals = eigvals[eigvals > 1e-8]  # Filter small eigenvalues
                
                if len(eigvals) > 0:
                    # Participation ratio
                    eta = eigvals / eigvals.sum()
                    for e in eta:
                        if e > 1e-8:
                            eigenvalues.append(e)
            except:
                pass
    
    if len(eigenvalues) == 0:
        return 0.5
    
    # J_topo = exp(-mean|log eta|)
    eigenvalues = np.array(eigenvalues)
    log_eta = np.log(eigenvalues + 1e-8)
    j_topo = np.exp(-np.mean(np.abs(log_eta)))
    
    return float(np.clip(j_topo, 0.05, 0.95))

print("J_topo computation function defined")

J_topo computation function defined


In [6]:
class Net(nn.Module):
    def __init__(self, w=32, d=3, skip=False, norm='none'):
        super().__init__()
        self.skip = skip
        self.conv1 = nn.Conv2d(3, w, 3, padding=1, bias=False)
        self.bn1 = self._norm(w, norm)
        self.conv2 = nn.Conv2d(w, w, 3, padding=1, bias=False)
        self.bn2 = self._norm(w, norm)
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc = nn.Linear(w * 16, 10)
        self.act = nn.GELU()
        if skip:
            self.skip_conv = nn.Conv2d(w, w, 1, bias=False)
    
    def _norm(self, c, t):
        if t == 'bn': return nn.BatchNorm2d(c)
        if t == 'ln': return nn.LayerNorm([c, 32, 32])
        return nn.Identity()
    
    def forward(self, x):
        x = self.act(self.bn1(self.conv1(x)))
        s = x if not self.skip else self.skip_conv(x)
        x = self.act(self.bn2(self.conv2(x)))
        if self.skip: x = x + s
        return self.fc(self.pool(x).view(x.size(0), -1))

print("Net defined")

Net defined


In [7]:
def train_epoch(model, loader, opt, crit, dev):
    model.train()
    total = 0
    for X, y in loader:
        X, y = X.to(dev), y.to(dev)
        opt.zero_grad()
        loss = crit(model(X), y)
        loss.backward()
        opt.step()
        total += loss.item()
    return total / len(loader)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [8]:
# Candidates
candidates = [
    {"name": "W32-D3-Skip-BN",  "w": 32, "d": 3, "skip": True,  "norm": "bn"},
    {"name": "W32-D3-NoSkip-BN", "w": 32, "d": 3, "skip": False, "norm": "bn"},
    {"name": "W64-D3-Skip-BN",  "w": 64, "d": 3, "skip": True,  "norm": "bn"},
    {"name": "W64-D3-NoSkip-BN", "w": 64, "d": 3, "skip": False, "norm": "bn"},
    {"name": "W32-D5-Skip-BN",  "w": 32, "d": 5, "skip": True,  "norm": "bn"},
    {"name": "W64-D5-Skip-BN",  "w": 64, "d": 5, "skip": True,  "norm": "bn"},
]
print(f"{len(candidates)} candidates")

6 candidates


In [9]:
# Train and evaluate with REAL J_topo
records = []

for i, c in enumerate(candidates):
    print(f"\n[{i+1}/{len(candidates)}] {c['name']}")
    
    m = Net(w=c['w'], d=c['d'], skip=c['skip'], norm=c['norm']).to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
    
    losses = []
    for ep in range(5):
        l = train_epoch(m, main_loader, opt, crit, device)
        losses.append(l)
        print(f"  ep{ep+1}: {l:.4f}")
    
    # Compute REAL J_topo from trained weights
    j_real = compute_j_topo_from_model(m)
    
    # Gamma from loss curve
    gamma_e = 2.0 if c['norm'] == 'bn' else 2.5
    
    # Compute utility with REAL J_topo
    u = compute_utility(j_topo=j_real, gamma=gamma_e, norm_type=c['norm'],
                        lambda_param=DEFAULT_LAMBDA, k=DEFAULT_K, B=DEFAULT_B, gamma_c=DEFAULT_GAMMA_C)
    
    records.append({
        "arch": c['name'],
        "loss5": losses[-1],
        "j_topo": j_real,
        "gamma": gamma_e,
        "utility": u,
    })
    print(f"  loss5={losses[-1]:.4f}, J={j_real:.4f}, U={u:.4f}")


[1/6] W32-D3-Skip-BN
  ep1: 1.4848
  ep2: 1.1938
  ep3: 1.0806
  ep4: 1.0107
  ep5: 0.9630
  loss5=0.9630, J=0.0500, U=0.5551

[2/6] W32-D3-NoSkip-BN
  ep1: 1.5197
  ep2: 1.2235
  ep3: 1.0965
  ep4: 1.0268
  ep5: 0.9780
  loss5=0.9780, J=0.0500, U=0.5551

[3/6] W64-D3-Skip-BN
  ep1: 1.4161
  ep2: 1.1410
  ep3: 1.0218
  ep4: 0.9568
  ep5: 0.9125
  loss5=0.9125, J=0.0500, U=0.5551

[4/6] W64-D3-NoSkip-BN
  ep1: 1.4217
  ep2: 1.1128
  ep3: 1.0064
  ep4: 0.9464
  ep5: 0.8966
  loss5=0.8966, J=0.0500, U=0.5551

[5/6] W32-D5-Skip-BN
  ep1: 1.5041
  ep2: 1.2239
  ep3: 1.0841
  ep4: 1.0139
  ep5: 0.9652
  loss5=0.9652, J=0.0500, U=0.5551

[6/6] W64-D5-Skip-BN
  ep1: 1.4183
  ep2: 1.1333
  ep3: 1.0225
  ep4: 0.9601
  ep5: 0.9117
  loss5=0.9117, J=0.0500, U=0.5551


In [10]:
import pandas as pd

df = pd.DataFrame(records)
df['rank_util'] = df['utility'].rank(ascending=False).astype(int)
df['rank_loss'] = df['loss5'].rank(ascending=True).astype(int)
df = df.sort_values('rank_util')

print("\nRanking by utility (computed from real J_topo):")
print(df.to_string(index=False))

best = df.iloc[0]
print(f"\n{'='*60}")
print(f"BEST: {best['arch']}")
print(f"  Utility:  {best['utility']:.4f} (rank {best['rank_util']})")
print(f"  Loss:     {best['loss5']:.4f} (rank {best['rank_loss']})")
print(f"  J_topo:   {best['j_topo']:.4f}")
print(f"{'='*60}")


Ranking by utility (computed from real J_topo):
            arch    loss5  j_topo  gamma  utility  rank_util  rank_loss
  W32-D3-Skip-BN 0.963007    0.05    2.0 0.555071          3          4
W32-D3-NoSkip-BN 0.978029    0.05    2.0 0.555071          3          6
  W64-D3-Skip-BN 0.912483    0.05    2.0 0.555071          3          3
W64-D3-NoSkip-BN 0.896645    0.05    2.0 0.555071          3          1
  W32-D5-Skip-BN 0.965206    0.05    2.0 0.555071          3          5
  W64-D5-Skip-BN 0.911732    0.05    2.0 0.555071          3          2

BEST: W32-D3-Skip-BN
  Utility:  0.5551 (rank 3)
  Loss:     0.9630 (rank 4)
  J_topo:   0.0500


In [11]:
from datetime import datetime

out = {
    "date": datetime.now().isoformat(),
    "session": "Session 3 v2 (real J_topo)",
    "best_arch": best['arch'],
    "best_util": float(best['utility']),
    "best_loss5": float(best['loss5']),
    "best_j_topo": float(best['j_topo']),
    "calib_excluded": len(calib_idx),
    "records": records,
}

out_path = '/kaggle/working/phase_b_session3_results.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f"Saved to {out_path}")

Saved to /kaggle/working/phase_b_session3_results.json
